# Task 1 — SFT

## Setup

In [ ]:
%pip install -q \
  "transformers==5.13.1" \
  "trl==1.8.0" \
  "peft==0.19.1" \
  "bitsandbytes==0.49.2" \
  "datasets==5.0.0" \
  "accelerate==1.14.0" \
  "scikit-learn==1.7.2" \
  "scipy>=1.12,<2" \
  "pandas>=2.2,<3" \
  "safetensors>=0.5"

In [ ]:
import gc
import json
import os
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy.sparse import hstack
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

assert torch.cuda.is_available()

print(torch.cuda.get_device_name(0))
print(torch.__version__)

## Data

In [ ]:
def find_file(filename: str, folder: str | None = None) -> Path:
    roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/task1_sft_colab"),
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    candidates = []

    for root in roots:
        if folder is not None:
            candidates.append(root / folder / filename)
        candidates.append(root / filename)

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    for root in roots:
        if root.exists():
            matches = list(root.glob(f"**/{filename}"))
            if matches:
                return matches[0].resolve()

    raise FileNotFoundError(filename)


def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


TRAIN_PATH = find_file("kid_adult.jsonl", "data")
STYLE_TEST_PATH = find_file("public_test_style.jsonl", "data")
STYLE_CLF_PATH = find_file("style_clf.pkl", "metrics")

OUTPUT_DIR = Path("/content/outputs/task1_sft")
if not Path("/content").exists():
    OUTPUT_DIR = Path.cwd() / "outputs" / "task1_sft"

ADAPTER_DIR = OUTPUT_DIR / "adapter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_rows = read_jsonl(TRAIN_PATH)
style_test_rows = read_jsonl(STYLE_TEST_PATH)

assert len(train_rows) == 1489
assert len(style_test_rows) == 50
assert {"prompt", "kid", "adult"} <= train_rows[0].keys()
assert {"prompt", "kid", "adult", "fact"} <= style_test_rows[0].keys()

print(TRAIN_PATH)
print(STYLE_TEST_PATH)
print(STYLE_CLF_PATH)
print(len(train_rows), len(style_test_rows))

In [ ]:
with STYLE_CLF_PATH.open("rb") as file:
    style_bundle = pickle.load(file)

style_clf = style_bundle["clf"]
style_vecs = style_bundle["vecs"]


def style_probabilities(texts: list[str]) -> np.ndarray:
    features = hstack(
        [vectorizer.transform(texts) for vectorizer in style_vecs],
        format="csr",
    )
    probabilities = style_clf.predict_proba(features)
    simple_column = int(np.where(style_clf.classes_ == 1)[0][0])
    return probabilities[:, simple_column]


gold_kid_p = float(
    style_probabilities([row["kid"] for row in style_test_rows]).mean()
)
gold_adult_p = float(
    style_probabilities([row["adult"] for row in style_test_rows]).mean()
)

assert gold_kid_p > 0.9
assert gold_adult_p < 0.1

print(f"gold_kid={gold_kid_p:.6f}")
print(f"gold_adult={gold_adult_p:.6f}")

In [ ]:
train_dataset = Dataset.from_list(
    [
        {
            "prompt": [{"role": "user", "content": row["prompt"]}],
            "completion": [{"role": "assistant", "content": row["kid"]}],
        }
        for row in train_rows
    ]
)

print(train_dataset)

## Training

In [ ]:
for variable_name in ("trainer", "model"):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    seed=SEED,
    data_seed=SEED,
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,
    max_length=512,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

trainable_dtypes = {
    parameter.dtype
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
}

assert trainable_dtypes == {torch.float32}

train_result = trainer.train()

trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print(train_result.metrics)
print(ADAPTER_DIR)

## Generation

In [ ]:
model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

trainer.optimizer = None
trainer.lr_scheduler = None

gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "left"


@torch.inference_mode()
def generate_greedy(
    prompts: list[str],
    batch_size: int = 1,
    max_new_tokens: int = 160,
) -> list[str]:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    set_seed(SEED)

    answers = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]

        rendered = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for prompt in batch_prompts
        ]

        inputs = tokenizer(
            rendered,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=384,
        ).to("cuda")

        input_width = inputs["input_ids"].shape[1]

        generated = model.generate(
            **inputs,
            do_sample=False,
            num_beams=1,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        answers.extend(
            answer.strip()
            for answer in tokenizer.batch_decode(
                generated[:, input_width:],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )
        )

    return answers


test_prompts = [row["prompt"] for row in style_test_rows]
generated_answers = generate_greedy(test_prompts)

assert len(generated_answers) == 50
assert all(generated_answers)

print(generated_answers[0])

## Metric

In [ ]:
p_simple_each = style_probabilities(generated_answers)
p_simple_mean = float(p_simple_each.mean())


def task1_interval(value: float) -> tuple[str, str]:
    if value < 0.4:
        return "А", "< 0.4"
    if value < 0.7:
        return "Г", "0.4 – 0.7"
    if value < 0.9:
        return "Б", "0.7 – 0.9"
    return "В", "0.9 – 1.0"


answer_letter, answer_interval = task1_interval(p_simple_mean)

results = pd.DataFrame(
    {
        "prompt": test_prompts,
        "generated_answer": generated_answers,
        "P_simple": p_simple_each,
    }
)

results.to_csv(
    OUTPUT_DIR / "public_test_style_predictions.csv",
    index=False,
    encoding="utf-8",
)

print(f"TASK_1_METRIC_P_SIMPLE={p_simple_mean:.6f}")
print(f"TASK_1_INTERVAL={answer_interval}")
print(f"TASK_1_ANSWER={answer_letter}")

# Задача 2

## DPO-датасет

In [ ]:
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

assert ADAPTER_DIR.exists()
assert "p_simple_mean" in globals()

task1_p_simple = float(p_simple_mean)

dpo_dataset = Dataset.from_list(
    [
        {
            "prompt": [
                {
                    "role": "user",
                    "content": row["prompt"],
                }
            ],
            "chosen": [
                {
                    "role": "assistant",
                    "content": row["kid"],
                }
            ],
            "rejected": [
                {
                    "role": "assistant",
                    "content": row["adult"],
                }
            ],
        }
        for row in train_rows
    ]
)

assert len(dpo_dataset) == 1489

print(dpo_dataset)
print(dpo_dataset[0])

## загрузка результата SFT

In [ ]:
for variable_name in (
    "trainer",
    "model",
    "base_model",
    "dpo_model",
    "dpo_trainer",
):
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

base_model.config.use_cache = False

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

dpo_model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=True,
)

dpo_model.config.use_cache = False
dpo_model.train()

trainable_params, all_params = (
    dpo_model.get_nb_trainable_parameters()
)

print(f"trainable={trainable_params}")
print(f"all={all_params}")

## DPO-обучение

In [ ]:
DPO_OUTPUT_DIR = (
    OUTPUT_DIR.parent / "task2_dpo"
)

DPO_ADAPTER_DIR = (
    DPO_OUTPUT_DIR / "adapter"
)

DPO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

dpo_args = DPOConfig(
    output_dir=str(
        DPO_OUTPUT_DIR / "checkpoints"
    ),

    seed=SEED,
    data_seed=SEED,

    num_train_epochs=1,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,

    max_length=512,
    truncation_mode="keep_start",

    beta=0.1,
    loss_type="sigmoid",

    precompute_ref_log_probs=True,
    precompute_ref_batch_size=1,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

for parameter in (
    dpo_trainer.model.parameters()
):
    if parameter.requires_grad:
        parameter.data = (
            parameter.data.float()
        )

trainable_dtypes = {
    parameter.dtype
    for parameter
    in dpo_trainer.model.parameters()
    if parameter.requires_grad
}

assert trainable_dtypes == {
    torch.float32
}

assert (
    "ref"
    in dpo_trainer.model.peft_config
)

print(
    dpo_trainer.model.peft_config.keys()
)
print(trainable_dtypes)

dpo_result = dpo_trainer.train()

dpo_trainer.save_model(
    str(DPO_ADAPTER_DIR)
)

tokenizer.save_pretrained(
    str(DPO_ADAPTER_DIR)
)

print(dpo_result.metrics)
print(DPO_ADAPTER_DIR)

## генерация

In [ ]:
model = dpo_trainer.model

model.set_adapter("default")
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

dpo_trainer.optimizer = None
dpo_trainer.lr_scheduler = None

gc.collect()
torch.cuda.empty_cache()

tokenizer.padding_side = "left"

generated_answers_dpo = (
    generate_greedy(test_prompts)
)

assert len(
    generated_answers_dpo
) == 50

assert all(generated_answers_dpo)

print(generated_answers_dpo[0])

## итоговая метрика

In [ ]:
p_simple_dpo_each = (
    style_probabilities(
        generated_answers_dpo
    )
)

p_simple_dpo = float(
    p_simple_dpo_each.mean()
)


def task2_interval(
    value: float,
) -> tuple[str, str]:
    if value < 0.4:
        return "Г", "< 0.4"

    if value < 0.7:
        return "А", "0.4 – 0.7"

    if value < 0.9:
        return "В", "0.7 – 0.9"

    return "Б", "0.9 – 1.0"


task2_letter, task2_range = (
    task2_interval(p_simple_dpo)
)

results_dpo = pd.DataFrame(
    {
        "prompt": test_prompts,
        "generated_answer":
            generated_answers_dpo,
        "P_simple":
            p_simple_dpo_each,
    }
)

results_dpo.to_csv(
    DPO_OUTPUT_DIR
    / "public_test_style_predictions.csv",
    index=False,
    encoding="utf-8",
)

print(
    f"TASK_1_P_SIMPLE="
    f"{task1_p_simple:.6f}"
)

print(
    f"TASK_2_P_SIMPLE="
    f"{p_simple_dpo:.6f}"
)

print(
    f"TASK_2_DELTA="
    f"{p_simple_dpo - task1_p_simple:+.6f}"
)

print(
    f"TASK_2_INTERVAL="
    f"{task2_range}"
)

print(
    f"TASK_2_ANSWER="
    f"{task2_letter}"
)